# 00 · Recolección y procesamiento de datos (sección 6.1)

1. **Anonimización adicional:** la fuente se identifica solo como "una empresa de telefonía móvil". Se renombran
   las columnas con el nombre de la marca y se enmascara en los planes; se eliminan las banderas sensibles.
2. **Bitácora de limpieza:** faltantes, banderas constantes, duplicados y codificaciones.
3. **Población de estudio (5.2.1):** registros de personas naturales, con `FECHA_ACTUALIZACION` entre enero y
   octubre de 2026 y edad válida.
4. **Estratos por mes (5.2.2):** meta planificada frente a lo efectivamente disponible.

Salidas: `datos/base_trabajo.parquet` (9.000 registros limpios), `datos/elegibles.parquet` y `datos/r00_preparacion.json`.

In [1]:
import re
import numpy as np
import pandas as pd
import config
import catalogo as cat

crudo = pd.read_csv(config.CSV, low_memory=False)
print(f"Registros recibidos: {len(crudo):,} · columnas: {crudo.shape[1]}")

Registros recibidos: 9,000 · columnas: 127


## 1. Anonimización adicional

In [2]:
df = crudo.rename(columns=lambda c: c.replace("CLARO", "TELCOX"))
marca_en_planes = int(df["PLAN_DESC"].astype(str).str.contains("claro", case=False).sum())
df["PLAN_DESC"] = df["PLAN_DESC"].astype("string").str.replace(r"(?i)claro", "Operador", regex=True)
SENSIBLES = [cat.COL(b) for b in cat.EXCLUIR]          # citas, contenido adulto, salud, droguería
df = df.drop(columns=[c for c in SENSIBLES if c in df.columns])
print(f"Columnas renombradas: {sum('CLARO' in c for c in crudo.columns)} · planes con la marca enmascarados: "
      f"{marca_en_planes} · banderas sensibles eliminadas: {len(SENSIBLES)}")

Columnas renombradas: 10 · planes con la marca enmascarados: 524 · banderas sensibles eliminadas: 4


## 2. Bitácora de limpieza

In [3]:
ban = [c for c in df.columns if c.startswith("BAN_")]
constantes = [c for c in ban if df[c].nunique(dropna=False) <= 1]
texto_vacio = {"ESTRATO": "Se", "TECNOLOGIA_TERMINAL_TOP": "SIN TECNOLOGIA", "GENERO": "ND", "SEG_MACRO_AUDIENCIA": "Na"}
atributos = [c for c in df.columns if c not in ("ID_CLIENTE", "ID_LINEA")]
firma = pd.util.hash_pandas_object(df[atributos].astype("string").fillna("<NA>"), index=False)
bitacora = [
    {"tema": "Clave técnica", "detalle": f"ID_LINEA tiene {df.ID_LINEA.nunique():,} valores únicos en {len(df):,} registros: no hay líneas repetidas",
     "decision": "Se usa ID_LINEA como clave para detectar duplicados"},
    {"tema": "Filas con atributos idénticos", "detalle": f"{int(firma.duplicated().sum()):,} registros repiten todos los atributos de otro, pero con ID_LINEA distinta y en el mismo mes",
     "decision": "Se conservan: son líneas distintas con los mismos valores (no se eliminan observaciones válidas por compartir valores)"},
    {"tema": "Clientes con varias líneas", "detalle": f"{df.ID_CLIENTE.nunique():,} clientes únicos; {int((df.ID_CLIENTE.value_counts() > 1).sum()):,} tienen más de una línea",
     "decision": "La unidad de análisis es el registro de línea, no el cliente"},
    {"tema": "Banderas constantes", "detalle": f"{len(constantes)} de {len(ban)} banderas tienen un solo valor", "decision": "Se excluyen de la ciencia de datos; no afectan las 6 variables"},
    {"tema": "Estrato sin dato", "detalle": f"{(df.ESTRATO == 'Se').mean():.1%} de los registros traen 'Se'", "decision": "No se usa estrato"},
    {"tema": "Terminal sin dato", "detalle": f"{(df.TECNOLOGIA_TERMINAL_TOP == 'SIN TECNOLOGIA').mean():.1%} sin tecnología del equipo", "decision": "No se usan variables de terminal"},
    {"tema": "Región comercial", "detalle": f"{df.REGION_COMERCIAL_TOP.isna().mean():.0%} vacía", "decision": "Se usa el departamento"},
    {"tema": "Fechas", "detalle": "FECHA_ACTUALIZACION viene como año-mes (sin día); no hay registros de septiembre",
     "decision": "No se puede verificar el corte del 20 de octubre; se reporta la ausencia de septiembre"},
]
for v in config.VARIABLES:
    bitacora.append({"tema": f"Faltantes en {v}", "detalle": f"{int(df[v].isna().sum())} registros", "decision": "Se excluyen solo del cálculo de esa variable"})
pd.DataFrame(bitacora)

,tema,detalle,decision
0,Clave técnica,"ID_LINEA tiene 9,000 valores únicos en 9,000 r...",Se usa ID_LINEA como clave para detectar dupli...
1,Filas con atributos idénticos,890 registros repiten todos los atributos de o...,Se conservan: son líneas distintas con los mis...
2,Clientes con varias líneas,"7,747 clientes únicos; 263 tienen más de una l...","La unidad de análisis es el registro de línea,..."
3,Banderas constantes,35 de 88 banderas tienen un solo valor,Se excluyen de la ciencia de datos; no afectan...
4,Estrato sin dato,87.0% de los registros traen 'Se',No se usa estrato
5,Terminal sin dato,84.0% sin tecnología del equipo,No se usan variables de terminal
6,Región comercial,100% vacía,Se usa el departamento
7,Fechas,FECHA_ACTUALIZACION viene como año-mes (sin dí...,No se puede verificar el corte del 20 de octub...
8,Faltantes en ANTIGUEDAD_MESES,6 registros,Se excluyen solo del cálculo de esa variable
9,Faltantes en VAL_CFM,2 registros,Se excluyen solo del cálculo de esa variable


## 3. Población de estudio y criterios de elegibilidad
El documento no fija el rango de edad hasta verificar los valores. Al revisarlos, 1.867 cédulas tienen `EDAD = 0`
(dato faltante codificado como cero) y un registro tiene 114 años. Se fija el rango válido en 18–100 años.

In [4]:
print("EDAD por tipo de documento (cuántos registros tienen 0):")
print(df.assign(edad_cero=df.EDAD == 0).groupby("TIPO_IDENTIFICACION").edad_cero.agg(["sum", "size"]))

en_periodo = df.FECHA_ACTUALIZACION.between(config.FECHA_INICIO, config.FECHA_FIN)
natural = df.TIPO_IDENTIFICACION != "Nit"
edad_valida = df.EDAD.between(config.EDAD_MIN, config.EDAD_MAX)
embudo = [
    {"paso": "Registros recibidos", "registros": len(df)},
    {"paso": "Actualizados entre enero y octubre de 2026", "registros": int(en_periodo.sum())},
    {"paso": "Personas naturales (sin NIT)", "registros": int((en_periodo & natural).sum())},
    {"paso": "Con edad válida (18–100 años)", "registros": int((en_periodo & natural & edad_valida).sum())},
]
elegibles = df[en_periodo & natural & edad_valida].copy()
pd.DataFrame(embudo)

EDAD por tipo de documento (cuántos registros tienen 0):
                      sum  size
TIPO_IDENTIFICACION            
Cedula               1867  7078
Cedulaextranjero        4    23
Nit                  1877  1879
Pasaporte              19    20


,paso,registros
0,Registros recibidos,9000
1,Actualizados entre enero y octubre de 2026,9000
2,Personas naturales (sin NIT),7121
3,Con edad válida (18–100 años),5230


## 4. Estratos por mes: meta planificada frente a lo disponible

In [5]:
todos = pd.DataFrame({"mes": list(config.MESES)})
recibidos = df.FECHA_ACTUALIZACION.value_counts().rename("recibidos")
eleg_mes = elegibles.FECHA_ACTUALIZACION.value_counts().rename("elegibles")
estratos = todos.join(recibidos, on="mes").join(eleg_mes, on="mes").fillna(0).astype({"recibidos": int, "elegibles": int})
estratos["nombre"] = estratos.mes.map(config.MESES)
estratos["meta"] = config.META_POR_MES
estratos["prop_en_elegibles"] = (estratos.elegibles / estratos.elegibles.sum()).round(4)   # proporción del mes dentro de la muestra elegible (no es un peso poblacional)
estratos

,mes,recibidos,elegibles,nombre,meta,prop_en_elegibles
0,2026-01,1000,501,Enero,10000,0.0958
1,2026-02,1000,376,Febrero,10000,0.0719
2,2026-03,1000,455,Marzo,10000,0.0870
3,2026-04,1000,573,Abril,10000,0.1096
4,2026-05,1000,726,Mayo,10000,0.1388
5,2026-06,1000,531,Junio,10000,0.1015
6,2026-07,1000,733,Julio,10000,0.1402
7,2026-08,1000,947,Agosto,10000,0.1811
8,2026-09,0,0,Septiembre,10000,0.0000
9,2026-10,1000,388,Octubre,10000,0.0742


In [6]:
config.guardar(df, "base_trabajo")
config.guardar(elegibles, "elegibles")

# Base limpia para entregar (guía del proyecto, punto 2): los 5.230 registros elegibles sin identificadores.
# Es un archivo local: no se publica junto con el dashboard.
SIN_ID = ["ID_CLIENTE", "ID_LINEA", "COD_ORION"]
elegibles.drop(columns=[c for c in SIN_ID if c in elegibles.columns]).to_csv(
    config.DATOS / "base_limpia_elegibles.csv", index=False, encoding="utf-8-sig")
config.guardar_json({
    "recibidos": len(df), "columnas": df.shape[1], "marca_en_planes": marca_en_planes,
    "sensibles_eliminadas": [s[4:] for s in SENSIBLES], "constantes": [c[4:] for c in constantes],
    "n_banderas": len(ban), "clientes_unicos": int(df.ID_CLIENTE.nunique()),
    "clientes_multilinea": int((df.ID_CLIENTE.value_counts() > 1).sum()),
    "atributos_identicos": int(firma.duplicated().sum()),
    "edad_cero_cedula": int(((df.EDAD == 0) & (df.TIPO_IDENTIFICACION == "Cedula")).sum()),
    "edad_mayor_100": int((df.EDAD > 100).sum()), "nit": int((~natural).sum()),
    "bitacora": bitacora, "embudo": embudo, "estratos": estratos.to_dict("records"),
    "postpago_elegibles": float((elegibles.TIPO_LINEA == "Línea Postpago").mean()),
    "faltantes": {v: int(elegibles[v].isna().sum()) for v in config.VARIABLES},
}, "r00_preparacion")
print(f"Elegibles: {len(elegibles):,} registros ({(elegibles.TIPO_LINEA == 'Línea Postpago').mean():.1%} postpago)")

Elegibles: 5,230 registros (98.5% postpago)
